In [ ]:
# Cell 1: Create and use virtualenv inside Colab
from pathlib import Path
import logging
import os
import shutil
import subprocess
import sys

CELL_NAME = "cell_01_setup"
CELL_REV = "2026-04-18-r4"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()  # delete old log on rerun

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        stderr_tail = (result.stderr or "").strip()[-1500:]
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{stderr_tail}")
    return result

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
REPO_URL = "https://github.com/kareemkamal10/NAMAA-Egyptian-Voice.git"
VENV_DIR = Path("/content/namaa-venv")
VENV_PYTHON = None

logger.info("Start Cell 1")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 1 revision: {CELL_REV}")
logger.info(f"Cell 1 revision: {CELL_REV}")

try:
    logger.info("Installing system packages for the isolated Python runtime")
    run_cmd(["apt-get", "update"] )
    run_cmd(["apt-get", "install", "-y", "python3.11", "python3.11-venv", "git-lfs"] )

    if not PROJECT_DIR.exists():
        logger.info("Cloning repository")
        run_cmd(["git", "clone", REPO_URL, str(PROJECT_DIR)])
    else:
        logger.info("Repository already exists; pulling latest changes")
        run_cmd(["git", "pull", "--ff-only"], cwd=str(PROJECT_DIR))

    run_cmd(["git", "lfs", "install"], cwd=str(PROJECT_DIR))
    run_cmd(["git", "lfs", "pull"], cwd=str(PROJECT_DIR))

    preferred_python = shutil.which("python3.11") or shutil.which("python3.10") or sys.executable

    if preferred_python is None:
        raise RuntimeError("No usable Python interpreter found to build the virtualenv.")

    logger.info(f"Using Python for virtualenv: {preferred_python}")
    print(f"Virtualenv Python: {preferred_python}")

    run_cmd([sys.executable, "-m", "pip", "install", "-U", "virtualenv"] )
    if VENV_DIR.exists():
        logger.info("Removing old virtualenv")
        shutil.rmtree(VENV_DIR)

    run_cmd([sys.executable, "-m", "virtualenv", "--python", preferred_python, str(VENV_DIR)])

    if os.name == "nt":
        VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
    else:
        VENV_PYTHON = VENV_DIR / "bin" / "python"

    logger.info(f"VENV_PYTHON={VENV_PYTHON}")
    print(f"VENV Python: {VENV_PYTHON}")

    run_cmd([str(VENV_PYTHON), "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], cwd=str(PROJECT_DIR))
    run_cmd([str(VENV_PYTHON), "-m", "pip", "install", "-r", "requirements.txt"], cwd=str(PROJECT_DIR))
    logger.info("requirements installed successfully inside virtualenv")
except Exception:
    logger.exception("Cell 1 failed")
    raise

logger.info("End Cell 1")


In [ ]:
# Cell 2: Runtime check inside the virtualenv
from pathlib import Path
import logging
import os
import subprocess
import textwrap

CELL_NAME = "cell_02_runtime_check"
CELL_REV = "2026-04-18-r5"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        stderr_tail = (result.stderr or "").strip()[-1500:]
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{stderr_tail}")
    return result

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
if os.name == "nt":
    VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
else:
    VENV_PYTHON = VENV_DIR / "bin" / "python"

def run_python_script(script_content, cwd=None):
    script_dir = PROJECT_DIR / "tmp_scripts"
    script_dir.mkdir(parents=True, exist_ok=True)
    script_file = script_dir / f"{CELL_NAME}.py"
    script_file.write_text(textwrap.dedent(script_content), encoding="utf-8")

    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(PROJECT_DIR) if not existing_pythonpath else f"{PROJECT_DIR}{os.pathsep}{existing_pythonpath}"
    return run_cmd([str(VENV_PYTHON), str(script_file)], cwd=cwd, env=env)

logger.info("Start Cell 2")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 2 revision: {CELL_REV}")
logger.info(f"Cell 2 revision: {CELL_REV}")

try:
    if not VENV_PYTHON.exists():
        raise FileNotFoundError("Virtualenv python not found. Run Cell 1 first.")
    if not (PROJECT_DIR / "app.py").exists():
        raise FileNotFoundError("app.py not found in project directory. Run Cell 1 first.")

    script = """
import sys
import torch

print('Python:', sys.version.replace(chr(10), ' '))
print('Executable:', sys.executable)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
"""
    run_python_script(script, cwd=str(PROJECT_DIR))
except Exception:
    logger.exception("Cell 2 failed")
    raise

logger.info("End Cell 2")


In [ ]:
# Cell 3: إدخال النص عبر TextField
from pathlib import Path
import logging
from IPython.display import display, Markdown
import ipywidgets as widgets

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

CELL_NAME = "cell_03_text_input"
CELL_REV = "2026-04-18-r8"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

logger.info("Start Cell 3")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 3 revision: {CELL_REV}")
logger.info(f"Cell 3 revision: {CELL_REV}")

default_text = "انا سبت الشغل و راجع دلوقتي علي طول."
PROMPT_TEXT_WIDGET = widgets.Textarea(
    value=default_text,
    placeholder="اكتب النص الذي تريد تحويله لصوت هنا...",
    description="",
    layout=widgets.Layout(width="100%", height="120px"),
)

display(Markdown("### أدخل النص المراد توليده"))
display(PROMPT_TEXT_WIDGET)
print("اكتب النص في الحقل بالأعلى ثم انتقل للخلية التالية.")
logger.info("Text field displayed")
logger.info("End Cell 3")


In [ ]:
# Cell 4: رفع الريفرنس الصوتي من الجهاز (اختياري)
from pathlib import Path
import logging
from IPython.display import display, Markdown
import ipywidgets as widgets

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

CELL_NAME = "cell_04_optional_reference"
CELL_REV = "2026-04-18-r2"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

logger.info("Start Cell 4")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 4 revision: {CELL_REV}")
logger.info(f"Cell 4 revision: {CELL_REV}")

REFERENCE_UPLOAD_WIDGET = widgets.FileUpload(
    accept=".wav,.mp3,.flac,.ogg,.m4a",
    multiple=False,
    description="رفع ملف الريفرنس",
    button_style="primary",
)

display(Markdown("### (اختياري) ارفع ملف الريفرنس الصوتي من جهازك"))
display(REFERENCE_UPLOAD_WIDGET)
print("إذا تركت هذا الحقل بدون رفع ملف، سيستخدم النظام الصوت الافتراضي تلقائيًا.")
logger.info("Reference upload widget displayed")
logger.info("End Cell 4")


In [ ]:
# Cell 5: توليد الصوت
from pathlib import Path
import logging
import os
import subprocess
import textwrap

CELL_NAME = "cell_05_generate"
CELL_REV = "2026-04-18-r10"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

# =========================
# عدل القيم هنا (مثل واجهة السلايدر)
# =========================
EXAGGERATION = 0.5   # range: 0.25 -> 2.0
CFG_PACE = 0.5       # range: 0.0  -> 1.0
SEED = 0             # 0 = عشوائي
TEMPERATURE = 0.8    # range: 0.05 -> 5.0

def run_cmd(command, cwd=None, env=None):
    result = subprocess.run(command, cwd=cwd, env=env, text=True, capture_output=True)
    if result.stdout:
        logger.info(result.stdout.strip())
    if result.stderr:
        logger.info(result.stderr.strip())
    if result.returncode != 0:
        stderr_tail = (result.stderr or "").strip()[-1500:]
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}\n{stderr_tail}")
    return result

def extract_uploaded_audio(upload_widget):
    if upload_widget is None:
        return "", b""

    value = getattr(upload_widget, "value", None)
    if not value:
        return "", b""

    entry = None
    if isinstance(value, dict):
        if value:
            entry = next(iter(value.values()))
    elif isinstance(value, (list, tuple)):
        if len(value) > 0:
            entry = value[0]

    if not isinstance(entry, dict):
        return "", b""

    file_name = entry.get("name")
    content = entry.get("content")

    if not file_name and isinstance(entry.get("metadata"), dict):
        file_name = entry["metadata"].get("name")
    if content is None and isinstance(entry.get("metadata"), dict):
        content = entry["metadata"].get("content")

    if not file_name or content is None:
        return "", b""

    if isinstance(content, memoryview):
        content = content.tobytes()
    elif hasattr(content, "tobytes"):
        content = content.tobytes()
    elif isinstance(content, bytearray):
        content = bytes(content)
    elif not isinstance(content, bytes):
        content = bytes(content)

    return str(file_name), content

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
VENV_DIR = Path("/content/namaa-venv")
if os.name == "nt":
    VENV_PYTHON = VENV_DIR / "Scripts" / "python.exe"
else:
    VENV_PYTHON = VENV_DIR / "bin" / "python"

def run_python_script(script_content, cwd=None):
    script_dir = PROJECT_DIR / "tmp_scripts"
    script_dir.mkdir(parents=True, exist_ok=True)
    script_file = script_dir / f"{CELL_NAME}.py"
    script_file.write_text(textwrap.dedent(script_content), encoding="utf-8")

    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(PROJECT_DIR) if not existing_pythonpath else f"{PROJECT_DIR}{os.pathsep}{existing_pythonpath}"
    return run_cmd([str(VENV_PYTHON), str(script_file)], cwd=cwd, env=env)

logger.info("Start Cell 5")
print(f"Logging to: {LOG_FILE}")
print(f"Cell 5 revision: {CELL_REV}")
logger.info(f"Cell 5 revision: {CELL_REV}")

try:
    if not VENV_PYTHON.exists():
        raise FileNotFoundError("Virtualenv python not found. Run Cell 1 first.")
    if not (PROJECT_DIR / "app.py").exists():
        raise FileNotFoundError("app.py not found in project directory. Run Cell 1 first.")

    prompt_widget = globals().get("PROMPT_TEXT_WIDGET")
    if prompt_widget is not None:
        text_input_value = str(prompt_widget.value).strip()
    else:
        text_input_value = str(globals().get("TEXT_INPUT", "")).strip()

    if not text_input_value:
        raise ValueError("الرجاء كتابة النص في حقل الخلية 3.")

    if not (0.25 <= float(EXAGGERATION) <= 2.0):
        raise ValueError("قيمة EXAGGERATION لازم تكون بين 0.25 و 2.0")
    if not (0.0 <= float(CFG_PACE) <= 1.0):
        raise ValueError("قيمة CFG_PACE لازم تكون بين 0.0 و 1.0")
    if not (0.05 <= float(TEMPERATURE) <= 5.0):
        raise ValueError("قيمة TEMPERATURE لازم تكون بين 0.05 و 5.0")

    seed_value = int(SEED)
    if seed_value < 0:
        raise ValueError("قيمة SEED لازم تكون 0 أو رقم موجب")

    upload_widget = globals().get("REFERENCE_UPLOAD_WIDGET")
    upload_name, upload_bytes = extract_uploaded_audio(upload_widget)

    audio_prompt_value = ""
    if upload_bytes:
        uploads_dir = PROJECT_DIR / "uploads"
        uploads_dir.mkdir(parents=True, exist_ok=True)
        safe_name = Path(upload_name).name
        saved_reference = uploads_dir / safe_name
        saved_reference.write_bytes(upload_bytes)
        audio_prompt_value = str(saved_reference)
        print(f"الريفرنس الصوتي: {saved_reference}")
        logger.info(f"Reference uploaded and saved: {saved_reference}")
    else:
        print("لا يوجد ريفرنس مرفوع. سيتم استخدام الصوت الافتراضي.")
        logger.info("No uploaded reference; using default")

    print("الإعدادات الحالية:")
    print("EXAGGERATION:", EXAGGERATION)
    print("CFG_PACE:", CFG_PACE)
    print("SEED:", seed_value)
    print("TEMPERATURE:", TEMPERATURE)
    logger.info(f"EXAGGERATION={EXAGGERATION}, CFG_PACE={CFG_PACE}, SEED={seed_value}, TEMPERATURE={TEMPERATURE}")

    script = f"""
from pathlib import Path
import soundfile as sf
import app

project_dir = Path('/content/NAMAA-Egyptian-Voice')
text_input = {text_input_value!r}
audio_prompt_raw = {audio_prompt_value!r}
audio_prompt = Path(audio_prompt_raw) if audio_prompt_raw else None
output_path = project_dir / 'outputs' / 'output.wav'
output_path.parent.mkdir(parents=True, exist_ok=True)

if audio_prompt is not None and not audio_prompt.exists():
    raise FileNotFoundError(f'Reference audio not found: {{audio_prompt}}')

# Ensure transformers can enable output_attentions during alignment analysis.
model = app.get_or_load_model()
tfmr = getattr(model, 'tfmr', None)
cfg = getattr(tfmr, 'config', None)
if cfg is not None:
    if hasattr(cfg, '_attn_implementation'):
        cfg._attn_implementation = 'eager'
    if hasattr(cfg, 'attn_implementation'):
        try:
            cfg.attn_implementation = 'eager'
        except Exception:
            pass
    print('attn_implementation:', getattr(cfg, '_attn_implementation', getattr(cfg, 'attn_implementation', 'unknown')))

sr, wav = app.generate_tts_audio(
    text_input=text_input,
    audio_prompt_path_input=(str(audio_prompt) if audio_prompt is not None else None),
    exaggeration_input={float(EXAGGERATION)!r},
    temperature_input={float(TEMPERATURE)!r},
    seed_num_input={seed_value!r},
    cfgw_input={float(CFG_PACE)!r},
 )
sf.write(output_path, wav, sr)
print('Saved:', output_path)
print('Sample rate:', sr)
print('Shape:', getattr(wav, 'shape', None))
"""
    run_python_script(script, cwd=str(PROJECT_DIR))
except Exception:
    logger.exception("Cell 5 failed")
    raise

logger.info("End Cell 5")


In [ ]:
# Cell 6: معاينة الصوت الناتج
from pathlib import Path
import logging
import os

CELL_NAME = "cell_06_preview"
LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOG_DIR / f"{CELL_NAME}.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

logger = logging.getLogger(CELL_NAME)
logger.handlers.clear()
logger.setLevel(logging.INFO)
fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

PROJECT_DIR = Path("/content/NAMAA-Egyptian-Voice")
OUTPUT_PATH = PROJECT_DIR / "outputs" / "output.wav"

logger.info("Start Cell 6")
print(f"Logging to: {LOG_FILE}")

try:
    from IPython.display import Audio, display

    if OUTPUT_PATH.exists():
        display(Audio(str(OUTPUT_PATH), autoplay=False))
        print("Audio preview ready.")
        logger.info("Audio preview displayed")
    else:
        print("Output file not found. Run Cell 5 first.")
        logger.warning("Output file not found")
except Exception:
    logger.exception("Cell 6 failed")
    raise

logger.info("End Cell 6")
